<center><a href="https://www.nvidia.com/dli"> <img src="images/DLI_Header.png" alt="Header" style="width: 400px;"/> </a></center>

# 2. 美國手語資料集的圖像分類(Image Classification of an American Sign Language Dataset)

在這個章節中，我們將使用不同的資料集來執行上一章節中觀察到的資料準備、模型建立和模型訓練步驟。資料集中的圖像是美國手語([American Sign Language](http://www.asl.gs/))。


## 2.1 目標

* 準備圖像資料以供訓練
* 創建並編譯一個簡單的圖像分類模型
* 訓練圖像分類模型並觀察結果

In [ ]:
import torch.nn as nn
import pandas as pd
import torch
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.is_available()

## 2.2 美國手語資料集(American Sign Language Dataset)

美國手語字母表([American Sign Language alphabet](http://www.asl.gs/) )包含26個字母，其中兩個字母（j和z）需要動作，因此不包含在訓練資料集中。


<img src="./images/asl.png" style="width: 600px;">

### 2.2.1 Kaggle

這個資料集可從[Kaggle](http://www.kaggle.com)網站取得，Kaggle是一個很棒的地方，可以找到許多資料集和其他深度學習資源。除了提供資料集和類似這些Notebook的「核心」(kernels)外，Kaggle還舉辦競賽，您可以參與競爭，與他人一起訓練高準確度的模型。

如果您想練習或查看許多深度學習專案的範例，Kaggle是一個很好的參考網站。


## 2.3 載入資料

T這個資料集不能像MNIST一樣透過TorchVision取得，因此讓我們學習如何載入自訂資料。到本章節結束時，我們將會有`x_train`、`y_train`、`x_valid`和`y_valid`變數。


### 2.3.1 讀取資料

手語資料集以[CSV](https://en.wikipedia.org/wiki/Comma-separated_values) (Comma Separated Values)格式儲存，與Microsoft Excel和Google Sheets後面的資料結構相同。它是一個帶有標籤的行列格，如在[train](data/asl_data/sign_mnist_train.csv)和[valid](data/asl_data/sign_mnist_valid.csv)資料集所見（可能需要一點時間載入）。

要載入和操作資料，我們將使用名為[Pandas](https://pandas.pydata.org/)的函式庫，這是一個用於載入和操作資料的高效工具。我們將讀取CSV檔並轉換成名為[DataFrame](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html)的格式。


Pandas有一個[read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)方法，該方法預期一個CSV檔，並返回一個DataFrame：


In [ ]:
train_df = pd.read_csv("data/asl_data/sign_mnist_train.csv")
valid_df = pd.read_csv("data/asl_data/sign_mnist_valid.csv")

### 2.3.2 探索資料

讓我們看看我們的資料。我們可以使用[head](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.head.html)方法印出DataFrame的前幾行。每一行代表一張圖像，包含一個`label`欄位，以及代表圖像中每個像素值的784個值，就像MNIST資料集一樣。請注意，標籤目前是數值，而不是字母表中的字母：


In [ ]:
train_df.head()

### 2.3.3 提取標籤

讓我們將訓練和驗證標籤儲存到`y_train`和`y_valid`變數中。我們可以使用[pop](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pop.html)方法從DataFrame中移除一個欄位，並將移除的值指派給變數。


In [ ]:
y_train = train_df.pop('label')
y_valid = valid_df.pop('label')
y_train

### 2.3.4 提取圖像

接下來，讓我們將訓練和驗證圖像儲存到`x_train`和`x_valid`變數中。這裡我們創建這些變數：


In [ ]:
x_train = train_df.values
x_valid = valid_df.values
x_train

### 2.3.5 總結訓練和驗證資料

我們現在有27,455張圖像，每張圖像有784個像素，供訓練使用...


In [ ]:
x_train.shape

...以及它們相對應的標籤：


In [ ]:
y_train.shape

對於驗證，我們有7,172張圖像...

In [ ]:
x_valid.shape

...以及它們相對應的標籤：


In [ ]:
y_valid.shape

## 2.4  視覺化(Visualizing)資料

要視覺化圖像，我們將再次使用[matplotlib](https://matplotlib.org/) 函式庫。我們不需要關心視覺化的細節，但如果有興趣，您可以稍後學習更多關於matplotlib的內容。

請注意，我們需要將資料從目前的1D形狀（784個像素）重塑為2D形狀（28x28像素），以便理解圖像：


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(40,40))

num_images = 20
for i in range(num_images):
    row = x_train[i]
    label = y_train[i]

    image = row.reshape(28,28)
    plt.subplot(1, num_images, i+1)
    plt.title(label, fontdict={'fontsize': 30})
    plt.axis('off')
    plt.imshow(image, cmap='gray')

### 2.4.1  正規化(Normalize)圖像資料

就像我們對MNIST資料集所做的那樣，我們將正規化(Normalize)圖像資料，這意味著它們的像素值將不再在0到255之間：


In [ ]:
x_train.min()

In [ ]:
x_train.max()

在前面的課程中，我們使用了[ToTensor](https://pytorch.org/vision/main/generated/torchvision.transforms.ToTensor.html)，但我們也可以在將資料轉換為張量之前修改它。


In [ ]:
x_train = train_df.values / 255
x_valid = valid_df.values / 255

### 2.4.2 自訂資料集


我們可以使用PyTorch的[Dataset](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)工具來創建自己的資料集。`__init__`將在類別初始化時執行一次。`__getitem__`返回我們的圖像和標籤。

由於我們的資料集夠小，我們可以將其儲存在GPU上以加快處理速度。在前面的課程中，我們在每個批次中抽取資料時將其發送到GPU。這裡，我們將在`__init__`函式中將其發送到GPU。


In [ ]:
class MyDataset(Dataset):
    def __init__(self, x_df, y_df):
        self.xs = torch.tensor(x_df).float().to(device)
        self.ys = torch.tensor(y_df).to(device)

    def __getitem__(self, idx):
        x = self.xs[idx]
        y = self.ys[idx]
        return x, y

    def __len__(self):
        return len(self.xs)


自訂的PyTorch資料集與預建的資料集一樣。它應該被傳遞給[DataLoader](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html#preparing-your-data-for-training-with-dataloaders)以進行模型訓練。



In [ ]:
BATCH_SIZE = 32

train_data = MyDataset(x_train, y_train)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
train_N = len(train_loader.dataset)

In [ ]:
valid_data = MyDataset(x_valid, y_valid)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)
valid_N = len(valid_loader.dataset)



我們可以使用以下程式碼驗證DataLoader是否按預期運作。我們將使DataLoader可疊代([iterable](https://docs.python.org/3/library/functions.html#iter))，並呼叫[next](https://docs.python.org/3/library/functions.html#next)來抽取第一個資料。


In [ ]:
train_loader

嘗試執行以下程式碼幾次。每次執行的值應該會改變。

In [ ]:
batch = next(iter(train_loader))
batch

請注意，批次有兩個值。第一個是我們的`x`，第二個是我們的`y`。每個的第一維度應該有`32`個值，這是`batch_size`。

In [ ]:
batch[0].shape

In [ ]:
batch[1].shape

## 2.5 建立模型

我們已經建立了 DataLoaders，現在是時候來建立我們的模型了。

#### 實作練習

在這個練習中，我們將建立一個序列模型(sequential model)。就像上次一樣，建立一個具有以下特徵的模型：
* 包含一個展平層(flatten layer)。
* 包含一個密集輸入層(dense input layer)。此層應包含 512 個神經元，並使用 `relu` 活化函式(Activation Function)。
* 包含第二個密集層(dense layer)，具有 512 個神經元，並使用 `relu` 活化函式(Activation Function)。
* 包含一個密集輸出層(dense output layer)，其神經元數量應等於分類的類別數。

我們將定義一些變數來開始：

In [ ]:
input_size = 28 * 28
n_classes = 24


請在下面的程式碼區塊(Cell)中進行操作，創建一個 `model` 變數來儲存模型。我們已匯入 [Sequental](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) 模型類別(class)以及 [Linear](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) 層類別(class)來幫助您開始。點擊下方 '...' 以獲得提示：

In [ ]:
model = nn.Sequential(

)

#### 解答

In [ ]:
# SOLUTION
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 512),  # Input
    nn.ReLU(),  # Activation for input
    nn.Linear(512, 512),  # Hidden
    nn.ReLU(),  # Activation for hidden
    nn.Linear(512, n_classes)  # Output
)

這次，我們將結合編譯模型並將其送到 GPU 的步驟：

In [ ]:
model = torch.compile(model.to(device))
model

由於對這些美國手語(ASL)圖像進行分類與對 MNIST 手寫數字進行分類相似，我們將使用相同的 損失函式(`loss function`) ([Categorical CrossEntropy](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)) 和 優化器(`optimizer`) ([Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html))。`nn.CrossEntropyLoss` 包含 `softmax` 函式，相對於預測概率(probabilities)，其在傳遞類別索引(class indices)時計算速度更快。

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters())

## 2.6 訓練模型

這次，我們將更詳細地查看 `train` 和 `validate` 函式。

### 2.6.1 訓練函式 (Train Function)


此程式碼幾乎與之前的 Notebook 相同，但我們不再將 `x` 和 `y` 傳送到 GPU，因為 `DataLoader` 已經完成了這項工作。

在迴圈處理全部 DataLoader中的資料之前，我們會將模型設置為 [model.train](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.train)，以確保其參數可以更新。為了讓我們更容易追蹤訓練進度，我們會記錄總體的 損失(`loss`) 和 準確度(`accuracy`)。

接著，對於在`train_loader`中每個批次(batch)的資料，我們會：

1. 從模型獲得一個 `output` 預測。
2. 使用 優化器(`optimizer`) 的 [zero_grad](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html) 函式將梯度設置為零。
3. 使用我們的 損失函式(`loss function`) 計算損失。
4. 使用 [backward](https://pytorch.org/docs/stable/generated/torch.Tensor.backward.html) 計算梯度。
5. 使用 優化器(`optimizer`) 的 [step](https://pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html) 函式更新模型參數。

更新總體的 損失(`loss`) 和 準確度(`accuracy`)。

In [ ]:
def train():
    loss = 0
    accuracy = 0

    model.train()
    for x, y in train_loader:
        output = model(x)
        optimizer.zero_grad()
        batch_loss = loss_function(output, y)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output, y, train_N)
    print('Train - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

### 2.6.2 驗證函式 (Validate Function)


在驗證期間，模型不會進行學習，因此 `validate` 函式比上述的 `train` 函式更簡單。

主要的差異是我們會將模型透過([model.evaluate](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval))設置為評估模式(evaluation mode)，以防止模型更新任何參數。

In [ ]:
def validate():
    loss = 0
    accuracy = 0

    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            output = model(x)

            loss += loss_function(output, y).item()
            accuracy += get_batch_accuracy(output, y, valid_N)
    print('Valid - Loss: {:.4f} Accuracy: {:.4f}'.format(loss, accuracy))

### 2.6.3 計算準確度 (Calculating the Accuracy)

`train` 和 `validate` 函式都使用了 `get_batch_accuracy`，但我們尚未在此 Notebook 中定義該函式

#### 實作練習

The function below has three `FIXME`s. Each one corresponds to the functions input arguments. Can you replace each FIXME with the correct argument?

It may help to view the documentation for [argmax](https://pytorch.org/docs/stable/generated/torch.argmax.html), [eq](https://pytorch.org/docs/stable/generated/torch.eq.html), and [view_as](https://pytorch.org/docs/stable/generated/torch.Tensor.view_as.html).

以下函式包含三個 `FIXME`。每一個都對應於函式的輸入參數。您能否用正確的參數替換每個 `FIXME`？

查看以下文件可能會有所幫助：[argmax](https://pytorch.org/docs/stable/generated/torch.argmax.html)、[eq](https://pytorch.org/docs/stable/generated/torch.eq.html)、以及 [view_as](https://pytorch.org/docs/stable/generated/torch.Tensor.view_as.html)。

In [ ]:
def get_batch_accuracy(output, y, N):
    pred = FIXME.argmax(dim=1, keepdim=True)
    correct = pred.eq(FIXME.view_as(pred)).sum().item()
    return correct / FIXME

#### 解答

點擊下方的 `...` 查看解答。

In [ ]:
# SOLUTION
def get_batch_accuracy(output, y, N):
    pred = output.argmax(dim=1, keepdim=True)
    correct = pred.eq(y.view_as(pred)).sum().item()
    return correct / N

### 2.6.3 訓練迴圈 (Training Loop)

讓我們把所有步驟結合起來！執行以下程式碼區塊(Cell)，以訓練資料 20 次 (`epochs`)。

In [ ]:
epochs = 20

for epoch in range(epochs):
    print('Epoch: {}'.format(epoch))
    train()
    validate()

### 2.6.4 討論：執行結果如何？


我們可以看到訓練準確度取得了相當好的結果，但驗證準確度並沒有那麼高。這裡發生了什麼？

思考一下再點擊下方的 `...` 來查看答案。

`# SOLUTION`

這是一個模型學習對訓練資料進行分類，但在面對未曾訓練過的新資料時表現不佳的例子。本質上，它只是記住了資料集，而沒有獲得對問題的穩健和一般性理解。這是一個常見的問題，稱為過擬合(*overfitting*)。我們將在接下來的兩堂課中討論過擬合，以及一些解決方法。

## 2.7 總結


在本節中，您建立了自己的神經網路(neural network)來執行相當準確的圖像分類(image classification)。恭喜！

此時，我們應該對載入資料（包括標籤(Labels)）、準備資料、創建模型，然後用準備好的資料訓練模型的過程有些熟悉了。

### 2.7.1 清除記憶體
在繼續之前，請執行以下程式碼區塊(Cell)來清理 GPU 記憶體。這是繼續下一個 Notebook 所必需的。

In [ ]:
import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

### 2.7.2 下一步


現在您已經建立了一些非常基本且有效的模型，我們將開始學習更複雜的模型，包括卷積神經網路(*Convolutional Neural Networks*)。

<center><a href="https://www.nvidia.com/dli"> <img src="images/DLI_Header.png" alt="Header" style="width: 400px;"/> </a></center>